# Kerr ell=4 decoupled hybrid -- reproduce `phase_c_l4_decoupled` (2026-07-04)

Reproduces the exact model documented in `kerr/configs/hybrid_kerr_l4_decoupled.yaml`
(k8 order-2 prior in, order-4 Richardson target, ell=4, mode-selective m1-m5 consensus
ensemble at scri). Drive-backed and resumable: every corpus split and the trained model
are saved to `MyDrive/kerr_l4/`, so a disconnect never loses work.

**PILOT vs FULL** is one switch in Cell 5:
- pilot (default): `N_TRAIN,N_VAL,N_TEST = 128,64,64` -- a real but small run for a preview
  and a go/no-go decision (~1 h; corpus build is the long pole, ~100 s/sample CPU).
- full (exact reproduction): set `128,64,64 -> 1024,128,128` and re-run from Cell 6.

**Cell 8 prints the decision table** (field / Mw / tau by spin, spread, sel_dist, gate);
**Cell 9 shows all 7 figures**. Run top-to-bottom; never use File -> Save a copy in GitHub.

In [ ]:
# Cell 2 -- mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/kerr_l4'
os.makedirs(DRIVE, exist_ok=True)
print('Drive work dir:', DRIVE)
!ls -lh {DRIVE} 2>/dev/null || echo '(empty - first run)'

In [ ]:
# Cell 3 -- sanity: GPU / CPU / RAM  (corpus build is CPU-bound; more vCPUs = faster)
!nvidia-smi
!nproc
!free -g | head -2

In [ ]:
# Cell 4 -- clone + pinned deps + allocator flag (set BEFORE torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MPLBACKEND'] = 'Agg'
%cd /content
!rm -rf QNM_Project_Fresh
!git clone --depth 1 https://github.com/ScreaM835/QNM_Project_Fresh.git
REPO = '/content/QNM_Project_Fresh'
%cd {REPO}
!pip -q install "neuraloperator==2.0.0"
import torch; print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
# Cell 5 -- config: PILOT vs FULL + paths
# pilot = a real but small run (preview + decision). FULL exact repro = 1024,128,128.
N_TRAIN, N_VAL, N_TEST = 128, 64, 64      # <-- set to 1024, 128, 128 for the exact reproduction
WORKERS = os.cpu_count() or 8             # corpus build parallelism (Linux fork)

REPO = '/content/QNM_Project_Fresh'
DRIVE = '/content/drive/MyDrive/kerr_l4'
DATA = f'{REPO}/kerr/outputs/phase_c_l4_decoupled'
OUT  = f'{DATA}/fno_hybrid_kerr'
CONFIG = 'kerr/configs/hybrid_kerr_l4_decoupled.yaml'
os.makedirs(DATA, exist_ok=True)
print(f'PILOT sizes: train {N_TRAIN}, val {N_VAL}, test {N_TEST}  (full = 1024/128/128)')
print(f'est. corpus build: ~{(N_TRAIN+N_VAL+N_TEST)*100/max(WORKERS,1)/60:.0f} min at {WORKERS} workers (~100 s/sample)')

In [ ]:
# Cell 6 -- corpus: reuse each split from Drive if present, else build once and SAVE to Drive.
# Teukolsky FD evolution (ell=4) on 4 grids -> CPU-bound; each split is saved as it completes,
# so a disconnect only costs the split in progress.
import os, shutil, time
%cd {REPO}
for S, N in [('train', N_TRAIN), ('val', N_VAL), ('test', N_TEST)]:
    repo_npz  = f'{DATA}/dataset_{S}.npz'
    drive_npz = f'{DRIVE}/dataset_{S}_n{N}.npz'
    if os.path.exists(drive_npz):
        print(f'[corpus] reuse {S} (n={N}) from Drive')
        shutil.copy(drive_npz, repo_npz)
    else:
        print(f'[corpus] building {S} n={N} ...', flush=True)
        t0 = time.time()
        !python kerr/scripts/build_kerr_dataset_lscan.py --ell 4 --split {S} \
            --coarse-n '2:401,4:201,8:101' --ks 2 4 8 --grid-order '1:4,2:4,4:4,8:2' \
            --n {N} --out {repo_npz} --workers {WORKERS}
        shutil.copy(repo_npz, drive_npz)
        print(f'[corpus] {S} built + saved to Drive in {(time.time()-t0)/60:.1f} min')
!ls -lh {DATA}/*.npz

In [ ]:
# Cell 7 -- train + eval + plots (one script). Saves the whole out_dir to Drive.
import os, shutil
%cd {REPO}
!MPLBACKEND=Agg PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python kerr/scripts/train_eval_hybrid_kerr.py --config {CONFIG}
drive_out = f'{DRIVE}/fno_hybrid_kerr'
os.makedirs(drive_out, exist_ok=True)
shutil.copytree(OUT, drive_out, dirs_exist_ok=True)
print('[train] out_dir saved -> Drive:', drive_out)

In [ ]:
# Cell 8 -- *** DECISION TABLE: everything we need, one place ***
import json
d = json.load(open(f'{OUT}/report.json'))
fr, mw, ta = d['field_rl2'], d['qnm_Mw_err_pct'], d['qnm_tau_err_pct']
sp, sd, g = d['qnm_spread_pct'], d['qnm_sel_dist'], d['acceptance_gate']
print('=' * 78)
print(f"  KERR ell=4 DECOUPLED HYBRID  (n_test={d['n_test']})   [medians]")
print('=' * 78)
print('OVERALL (prior -> hybrid, fine = extraction floor):')
print('  field rel-L2 : {:6.2f}% -> {:6.3f}%   (richardson target {:.3f}%)'.format(
    fr['prior']['median']*100, fr['hybrid']['median']*100, fr['richardson']['median']*100))
print('  Mw error     : {:6.2f}% -> {:6.3f}%   (fine {:.3f}%)'.format(
    mw['prior']['median'], mw['hybrid']['median'], mw['fine']['median']))
print('  tau error    : {:6.2f}% -> {:6.3f}%   (fine {:.3f}%)'.format(
    ta['prior']['median'], ta['hybrid']['median'], ta['fine']['median']))
print('  ensemble spread (hybrid) {:.2f}% | sel_dist (hybrid) {:.4f}'.format(
    sp['hybrid']['median'], sd['hybrid']['median']))
print('-' * 78)
print('BY SPIN (prior -> hybrid [fine]):')
hdr = '{:>12} | {:>2} | {:>16} | {:>18} | {:>18}'
print(hdr.format('a/M bin', 'n', 'field %', 'Mw %', 'tau %'))
for b in d['by_spin']:
    def f3(x):
        return '  nan' if x is None or x != x else '{:.2f}'.format(x)
    print(hdr.format(
        '[{:.2f},{:.2f}]'.format(*b['bin']), b['n'],
        '{:.1f}->{:.2f}'.format(b['field_rl2_prior']*100, b['field_rl2_hybrid']*100),
        '{}->{} [{}]'.format(f3(b['qnm_Mw_err_prior']), f3(b['qnm_Mw_err_hybrid']), f3(b['qnm_Mw_err_fine'])),
        '{}->{} [{}]'.format(f3(b['qnm_tau_err_prior']), f3(b['qnm_tau_err_hybrid']), f3(b['qnm_tau_err_fine']))))
print('-' * 78)
print('acceptance gate: field_pass={} qnm_pass={} OVERALL={}'.format(
    g['field_pass'], g['qnm_pass'], g['overall_pass']))
print('(gate thresholds are arbitrary CI lines; judge by the numbers above, not pass/fail)')
print('=' * 78)

In [ ]:
# Cell 9 -- show all 7 figures inline
from IPython.display import Image, display
import os
figs = f'{OUT}/figs'
for fn in ['hybrid_field_vs_spin.png', 'hybrid_qnm_vs_spin.png', 'hybrid_tau_vs_spin.png',
           'hybrid_ringdown_scri.png', 'hybrid_pointwise_error_hybrid.png',
           'hybrid_pointwise_error_baseline.png', 'hybrid_loss.png']:
    p = os.path.join(figs, fn)
    if os.path.exists(p):
        print(fn); display(Image(p))

In [ ]:
# Cell 10 -- zip report + per_sample + figures and download to the laptop
import os
!cd {OUT} && zip -r /content/kerr_l4_pilot.zip report.json per_sample.json figs history.json
from google.colab import files
files.download('/content/kerr_l4_pilot.zip')